In [2]:
from dotenv import load_dotenv
import os

import diqrng

load_dotenv()

True

In [7]:
def save_bits(filename: str, bitstring: str):
    bytestrings = [bitstring[i:i+8] for i in range(0, len(bitstring), 8)]
    ba = bytearray(int(byte, 2) for byte in bytestrings)
    with open(f"output/{filename}.bin", "wb") as file:
        file.write(bytes(ba))

# Demos

## Simulated Backend

To run on a simulated backend, pass `None` as the `token`.

Generate random numbers across two rounds and check CHSH violations across the other two.

In [3]:
try:
    diqrng.generate([0, 1, 0, 1, 0, 1, 0, 1], 4, token=None, bases=diqrng.BASES)
except diqrng.AbortGeneration as e:
    print(e)

Violations
[2.0, 2.4000000000000004, 2.2, 2.8]
Aborted. CHSH violations ([2.0, 2.4000000000000004, 2.2]) do not satisfy threshold (2.6627416997969524)


Due to noise, the CHSH inequality will not be violated. Repeat with increased uncertainty and increase the number of shots in check rounds to reduce variance.

In [4]:
uncertainty = 0.3
print("Producing random numbers with minimum CHSH violation:", diqrng.get_threshold(uncertainty))
bs = diqrng.generate([0, 1, 0, 1, 0, 1, 0, 1], 4, token=None, bases=diqrng.BASES, uncertainty=uncertainty, check_shots=10000)
save_bits("16_FakeKyiv_sim", bs)

Producing random numbers with minimum CHSH violation: 2.5798989873223332
Violations
[2.7598000000000003, 2.7618, 2.7228, 2.6856]
Generating  0  random numbers


## IBMQ Backend

To run on an IBMQ computer, pass your IMBQ API token to `token`. The script below assumes you have saved your token under `IMBQ_API_TOKEN` in a local `.env` file.

Note that generation rounds are placed next to one another for efficiecy purposes.

In [5]:
uncertainty = 0.3
print("Producing random numbers with minimum CHSH violation:", diqrng.get_threshold(uncertainty))
bs = diqrng.generate([0, 0, 0, 0, 1, 1, 1, 1], 4, token=os.getenv("IBMQ_API_TOKEN"), bases=diqrng.BASES, uncertainty=uncertainty, check_shots=10000, backend="ibm_brisbane")
save_bits("16_Brisbane_ibm", bs)

Producing random numbers with minimum CHSH violation: 2.5798989873223332
Violations
[2.6466, 2.692, 2.6778000000000004, 2.626]
Generating  0  random numbers


# Investigation

In [3]:
import numpy as np
import plotly.express as px
import pandas as pd
import time

## Time Efficiency

### Impact of parameters on the time taken to generate qubits.

In [7]:
range_of_pairs = range(1, 10)
range_of_shots = [1, 10, 50, 100, 500, 1000, 5000, 10000, 50000, 100000]
time_taken = {"pairs":[], "shots":[], "time":[]}


for num_pairs in range_of_pairs:
    circuit = diqrng.ChshCircuit(num_pairs=num_pairs, token=None)
    print(f"Number of pairs: {num_pairs}")
    if num_pairs > 7:
        range_of_shots.pop()
    for num_shots in range_of_shots:
        start = time.time()
        circuit.generate_numbers(num_shots=num_shots)
        end = time.time()
        print(f"Time taken for {num_shots} number of shots: {end - start}")
        time_taken["pairs"].append(num_pairs)
        time_taken["shots"].append(num_shots)
        time_taken["time"].append(end - start)

time_taken = pd.DataFrame(time_taken)

plot = px.line(time_taken, x="shots", y="time", color="pairs", title="Time taken to generate random numbers with different number of pairs and shots")
plot.write_image("output/time_generate.png")
plot.show()

Number of pairs: 1
Time taken for 1 number of shots: 0.6258060932159424
Time taken for 10 number of shots: 0.6184890270233154
Time taken for 50 number of shots: 0.6165401935577393
Time taken for 100 number of shots: 0.6206047534942627
Time taken for 500 number of shots: 0.6367759704589844
Time taken for 1000 number of shots: 0.6330888271331787
Time taken for 5000 number of shots: 0.5077252388000488
Time taken for 10000 number of shots: 0.6390631198883057
Time taken for 50000 number of shots: 0.6900529861450195
Time taken for 100000 number of shots: 0.6235430240631104
Number of pairs: 2
Time taken for 1 number of shots: 0.7786149978637695
Time taken for 10 number of shots: 0.6538000106811523
Time taken for 50 number of shots: 0.5043129920959473
Time taken for 100 number of shots: 0.6340177059173584
Time taken for 500 number of shots: 0.5024189949035645
Time taken for 1000 number of shots: 0.6263868808746338
Time taken for 5000 number of shots: 0.635138750076294
Time taken for 10000 numb

### Impact of parameters on the time taken to measure CHSH observables

In [9]:
range_of_pairs = range(1, 10)
range_of_shots = [1, 10, 50, 100, 500, 1000, 5000, 10000, 50000, 100000]
time_taken = {"pairs":[], "shots":[], "time":[]}


for num_pairs in range_of_pairs:
    circuit = diqrng.ChshCircuit(num_pairs=num_pairs, token=None)
    print(f"Number of pairs: {num_pairs}")
    if num_pairs > 6:
        range_of_shots.pop()
    for num_shots in range_of_shots:
        start = time.time()
        for basis in diqrng.BASES:
            circuit.measure_chsh_basis(basis, num_shots=num_shots)
        end = time.time()
        print(f"Time taken for {num_shots} number of shots: {end - start}")
        time_taken["pairs"].append(num_pairs)
        time_taken["shots"].append(num_shots)
        time_taken["time"].append(end - start)

time_taken = pd.DataFrame(time_taken)

plot = px.line(time_taken, x="shots", y="time", color="pairs", title="Time taken to measure CHSH observables with different number of pairs and shots")
plot.write_image("output/time_measure.png")
plot.show()

Number of pairs: 1
Time taken for 1 number of shots: 2.739427089691162
Time taken for 10 number of shots: 2.3602800369262695
Time taken for 50 number of shots: 2.5227057933807373
Time taken for 100 number of shots: 2.3876659870147705
Time taken for 500 number of shots: 2.3972091674804688
Time taken for 1000 number of shots: 2.4720780849456787
Time taken for 5000 number of shots: 2.4382238388061523
Time taken for 10000 number of shots: 2.504610061645508
Time taken for 50000 number of shots: 2.6062662601470947
Time taken for 100000 number of shots: 2.734517812728882
Number of pairs: 2
Time taken for 1 number of shots: 3.4606311321258545
Time taken for 10 number of shots: 3.164252996444702
Time taken for 50 number of shots: 3.060056209564209
Time taken for 100 number of shots: 3.198827028274536
Time taken for 500 number of shots: 3.0418412685394287
Time taken for 1000 number of shots: 3.216289758682251
Time taken for 5000 number of shots: 3.096885919570923
Time taken for 10000 number of s

## Generating 10M qubits

In [5]:
num_pairs = 6
num_gen_shots = 16667
num_check_shots = 10000
num_gen_rounds = 100
num_check_rounds = 12
uncertainty = 0.3

random_indexes = np.random.choice(range(num_gen_rounds + num_check_rounds), num_check_rounds, replace=False)
rounds = [0 for _ in range(num_gen_rounds + num_check_rounds)]
for index in random_indexes:
    rounds[index] = 1
bases = [diqrng.BASES[0] for _ in range(num_check_rounds)]
for basis in diqrng.BASES[1:] * 3:
    random_index = np.random.choice(range(num_check_rounds), 1)[0]
    while bases[random_index] != diqrng.BASES[0]:
        random_index = (random_index + 1) % num_check_rounds
    bases[random_index] = basis

In [8]:
print("Producing random numbers with minimum CHSH violation:", diqrng.get_threshold(uncertainty))

generated_numbers = diqrng.generate(
    rounds,
    num_pairs,
    token=None,
    bases=bases,
    gen_shots=num_gen_shots,
    check_shots=num_check_shots,
    uncertainty=uncertainty
)

save_bits("10M_FakeKyiv_sim", generated_numbers)

Producing random numbers with minimum CHSH violation: 2.5798989873223332
Violations
[2.774666666666667, 2.763466666666667, 2.7306, 2.6206, 2.7123333333333335, 2.7248]
Generating  1000020  random numbers
